# Colab EEG Training Notebook

Use this notebook to upload a zipped EEG dataset, load the code in this repo, and train the model on Colab.

- For BCI IV 2a, upload a zip that contains files like `A01T.gdf`, `A01E.gdf`, and `A01E.mat`.
- For EEGMMIDB, upload a zip that contains folders like `S001/`, `S002/`, etc.
- If you open the notebook outside the repo folder, the setup cell clones the repository automatically.

In [ ]:
# Install the small set of runtime dependencies used by the loaders and plots.
!pip -q install mne pyyaml scikit-learn matplotlib

In [ ]:
from pathlib import Path
import os
import random
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/Wasiq-Tariq11/BCI_test.git"
REPO_DIR = Path("BCI_test")

if not Path("bci2a_dataset.py").exists():
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL])
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())

In [ ]:
# Upload a single zip file containing your dataset.
try:
    from google.colab import files
    uploaded = files.upload()
except Exception:
    uploaded = {}
    print("Not running in Colab. Set DATASET_ARCHIVE manually before continuing.")

DATASET_ARCHIVE = None
DATASET_EXTRACT_DIR = Path("/content/bci_dataset")
DATASET_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if uploaded:
    archive_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
    if archive_name is None:
        raise ValueError("Upload a .zip file containing the dataset.")
    DATASET_ARCHIVE = Path(archive_name)
    with zipfile.ZipFile(DATASET_ARCHIVE, "r") as zf:
        zf.extractall(DATASET_EXTRACT_DIR)
    print("Extracted dataset to:", DATASET_EXTRACT_DIR)
else:
    print("No archive uploaded yet. Set DATASET_ARCHIVE and extract it, then rerun this cell.")

In [ ]:
import yaml
import numpy as np
import torch
import matplotlib.pyplot as plt

from bci2a_dataset import build_bciv2a_loaders
from eegmmidb_dataset import build_eegmmidb_loaders
from eegnet_v4 import build_model

CONFIG_PATH = Path("Config.yaml")
with CONFIG_PATH.open("r") as f:
    cfg = yaml.safe_load(f)

DATASET_TYPE = "bciv2a"  # change to "eegmmidb" if you uploaded EEGMMIDB

def find_dataset_root(base_dir: Path, dataset_type: str) -> Path:
    base_dir = Path(base_dir)
    if dataset_type == "bciv2a":
        marker = next(base_dir.rglob("A01T.gdf"), None)
        if marker is None:
            raise FileNotFoundError("Could not find A01T.gdf inside the extracted archive.")
        return marker.parent
    marker = next(base_dir.rglob("S001R03.edf"), None)
    if marker is None:
        raise FileNotFoundError("Could not find S001R03.edf inside the extracted archive.")
    return marker.parent.parent

if DATASET_ARCHIVE is not None:
    dataset_root = find_dataset_root(DATASET_EXTRACT_DIR, DATASET_TYPE)
else:
    dataset_root = Path("/content/path-to-your-dataset")

if DATASET_TYPE == "bciv2a":
    cfg["paths"]["bciv2a_root"] = str(dataset_root)
else:
    cfg["paths"]["eegmmidb_root"] = str(dataset_root)
    if cfg["eegmmidb"].get("subjects") is None:
        cfg["eegmmidb"]["subjects"] = [1, 2]

print("Dataset type:", DATASET_TYPE)
print("Dataset root :", dataset_root)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

if DATASET_TYPE == "bciv2a":
    phase = "finetune"
    HELD_OUT_SUBJECT = 1
    train_loader, val_loader = build_bciv2a_loaders(cfg, subject=HELD_OUT_SUBJECT)
else:
    phase = "pretrain"
    train_loader, val_loader = build_eegmmidb_loaders(cfg)

model = build_model(cfg, phase=phase).to(device)
print("Device:", device)
print("Train batches:", len(train_loader), "Val batches:", len(val_loader))
xb, yb = next(iter(train_loader))
print("Example batch shape:", xb.shape, yb.shape)

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def mixup_data(x: torch.Tensor, y: torch.Tensor, alpha: float, device: str):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam

def mixup_criterion(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)

def build_optimizer(model: torch.nn.Module, cfg: dict, phase: str):
    if phase == "pretrain":
        pcfg = cfg["pretrain"]
        return torch.optim.AdamW(
            model.parameters(),
            lr=pcfg["lr"],
            weight_decay=pcfg["weight_decay"],
        )

    fcfg = cfg["finetune"]
    lr_groups = fcfg["lr_groups"]
    named_params = list(model.named_parameters())

    def params_with(prefixes):
        return [
            param for name, param in named_params
            if param.requires_grad and any(name.startswith(prefix) for prefix in prefixes)
        ]

    param_groups = []
    group_specs = [
        ("temporal_conv", ["temporal_conv", "bn1"]),
        ("depthwise_conv", ["depthwise_conv", "bn2"]),
        ("separable_conv", ["sep_depthwise", "sep_pointwise", "bn3"]),
        ("classifier", ["classifier"]),
    ]

    for group_name, prefixes in group_specs:
        group_params = params_with(prefixes)
        if group_params:
            param_groups.append({
                "params": group_params,
                "lr": lr_groups[group_name],
            })

    return torch.optim.AdamW(param_groups, weight_decay=fcfg["weight_decay"])

def run_training(model, train_loader, val_loader, cfg, phase, device):
    pcfg = cfg[phase]
    use_amp = bool(pcfg.get("mixed_precision", False) and device == "cuda")
    criterion = torch.nn.CrossEntropyLoss(label_smoothing=pcfg.get("label_smoothing", 0.0))
    optimizer = build_optimizer(model, cfg, phase)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    mixup_alpha = pcfg.get("mixup_alpha", 0.0)

    results_dir = Path(cfg["paths"]["results_dir"])
    results_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = results_dir / f"{phase}_best.pth"

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_val = -float("inf")
    stale_epochs = 0
    patience = pcfg["early_stopping"]["patience"]
    min_delta = pcfg["early_stopping"]["min_delta"]

    for epoch in range(1, pcfg["epochs"] + 1):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            if mixup_alpha > 0:
                x, y_a, y_b, lam = mixup_data(x, y, mixup_alpha, device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(x)
                if mixup_alpha > 0:
                    loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
                else:
                    loss = criterion(logits, y)

            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            if hasattr(model, "apply_max_norm"):
                model.apply_max_norm(max_norm=1.0)

            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            ref_targets = y_a if mixup_alpha > 0 else y
            correct += (preds == ref_targets).sum().item()
            total += x.size(0)

        train_loss = total_loss / max(total, 1)
        train_acc = correct / max(total, 1)

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                y = y.to(device)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    logits = model(x)
                    loss = criterion(logits, y)

                val_loss += loss.item() * x.size(0)
                val_correct += (logits.argmax(dim=1) == y).sum().item()
                val_total += x.size(0)

        val_loss = val_loss / max(val_total, 1)
        val_acc = val_correct / max(val_total, 1)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        print(
            f"Epoch {epoch:03d}/{pcfg['epochs']} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}"
        )

        if val_acc > best_val + min_delta:
            best_val = val_acc
            stale_epochs = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "history": history,
                "cfg": cfg,
                "val_acc": val_acc,
                "val_loss": val_loss,
            }, checkpoint_path)
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print(f"Early stopping after {epoch} epochs.")
                break

    best_ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(best_ckpt["model_state_dict"])
    return history, checkpoint_path

In [ ]:
seed = cfg[phase]["seed"]
set_seed(seed)
history, checkpoint_path = run_training(model, train_loader, val_loader, cfg, phase, device)
print("Best checkpoint saved to:", checkpoint_path)

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(epochs, history["train_acc"], label="train")
axes[1].plot(epochs, history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].legend()

fig.tight_layout()
plt.show()

try:
    from google.colab import files
    files.download(str(checkpoint_path))
except Exception:
    pass